In [27]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import json
import os

from pathlib import Path

In [28]:
def read_json(path: Path):
    with open(path, 'r', encoding='utf-8') as file:
        return json.load(file)

def read_xlsx(path: Path):
    return pd.read_excel(path)

def read_csv(path: Path):
    return pd.read_csv(path)

def read_txt(path: Path):
    with open(path, "r") as file:
        return [line.strip() for line in file]

In [50]:
config_home = Path('/home/karthikeya/Manas/edgepii/configs/edgepii_experiments_v2').resolve()
results_home = Path('/home/karthikeya/Manas/edgepii/results').resolve()

comparission_plan_path = config_home / 'experiment_plan.csv'
current_analysis = ['run_order_01_baselines.txt', 'run_order_02_learning_curves.txt']

datasets = ['ai4privacy', 'capid']
dataset_splits = ['validation', 'test', 'reddit']

In [51]:
comp_plan = read_csv(comparission_plan_path)

run_ids_det = []
run_ids_rel = []

for grp in current_analysis:
    for run in read_txt(config_home / grp):
        run_id = run.split('/')[-1].split('.')[0]
        exp_task = comp_plan.loc[comp_plan['run_id'] == run_id, 'task'].values[0]

        if exp_task == 'detection':
            run_ids_det.append(run_id)
        else:
            run_ids_rel.append(run_id)

comp_plan_det = comp_plan[comp_plan['run_id'].isin(run_ids_det)]
comp_plan_rel = comp_plan[comp_plan['run_id'].isin(run_ids_rel)]

In [52]:
runs_results = []

for run_id in run_ids_det:
    local_map = {'run_id': run_id}

    for dataset in datasets:
        for split in dataset_splits:
            result_json_path = results_home / f'{run_id}-{dataset}-{split}.json'

            if not result_json_path.exists():
                continue

            result = read_json(result_json_path)

            # print(result)
            # total_samples = result['documents']

            for abc in ['span', 'untyped_span']:
                local_map[f'{dataset}_{split}_{abc}_precision'] = result['metrics'][abc]['precision']
                local_map[f'{dataset}_{split}_{abc}_recall'] = result['metrics'][abc]['recall']
                local_map[f'{dataset}_{split}_{abc}_f1'] = result['metrics'][abc]['f1']

    runs_results.append(local_map)

runs_comp_plan_det = pd.DataFrame(runs_results)
comp_plan_det = comp_plan_det.merge(runs_comp_plan_det, on='run_id', how='left')

In [53]:
runs_results = []

for run_id in run_ids_rel:
    local_map = {'run_id': run_id}

    for dataset in datasets:
        for split in dataset_splits:
            result_json_path = results_home / f'{run_id}-{dataset}-{split}.json'

            if not result_json_path.exists():
                continue

            result = read_json(result_json_path)
            
            local_map[f'{dataset}_{split}_accuracy'] = result['metrics']['accuracy']
            local_map[f'{dataset}_{split}_macro_f1'] = result['metrics']['macro_f1']


    runs_results.append(local_map)

runs_comp_plan_rel = pd.DataFrame(runs_results)
comp_plan_rel = comp_plan_rel.merge(runs_comp_plan_rel, on='run_id', how='left')

### Which model performs best? - Detection

#### a) based on ai4privacy_validation_span_f1

In [46]:
metrics = ['span_f1', 'untyped_span_f1']

best_models = comp_plan_det.sort_values('ai4privacy_validation_span_f1', ascending=False).groupby('model', as_index=False).first()

result = best_models[['run_id', 'model'] + [f'ai4privacy_test_{metric}' for metric in metrics] + [f'capid_test_{metric}' for metric in metrics] + [f'capid_reddit_{metric}' for metric in metrics]].copy()
result.columns = ['run_id', 'Model', 'Typed-F1-ai4', 'Untyped-F1-ai4', 'Typed-F1-capid', 'Untyped-F1-capid', 'Typed-F1-reddit', 'Untyped-F1-reddit']
result.iloc[:, 2:] = result.iloc[:, 2:].round(4)

display(result)

,run_id,Model,Typed-F1-ai4,Untyped-F1-ai4,Typed-F1-capid,Untyped-F1-capid,Typed-F1-reddit,Untyped-F1-reddit
0,v2-bert-ai4-det-full-s42,bert,0.9759,0.9775,0.2324,0.3709,0.3221,0.3736
1,v2-distilbert-ai4-det-full-s42,distilbert,0.9750,0.9776,0.2579,0.3816,0.3705,0.3951
2,v2-minilm-ai4-det-full-s42,minilm,0.9733,0.9754,0.2413,0.3719,0.3286,0.3787


#### b) based on capid_validation_span_f1

In [47]:
metrics = ['span_f1', 'untyped_span_f1']

best_models = comp_plan_det.sort_values('capid_validation_span_f1', ascending=False).groupby('model', as_index=False).first()

result = best_models[['run_id', 'model'] + [f'ai4privacy_test_{metric}' for metric in metrics] + [f'capid_test_{metric}' for metric in metrics] + [f'capid_reddit_{metric}' for metric in metrics]].copy()
result.columns = ['run_id', 'Model', 'Typed-F1-ai4', 'Untyped-F1-ai4', 'Typed-F1-capid', 'Untyped-F1-capid', 'Typed-F1-reddit', 'Untyped-F1-reddit']
result.iloc[:, 2:] = result.iloc[:, 2:].round(4)

display(result)

,run_id,Model,Typed-F1-ai4,Untyped-F1-ai4,Typed-F1-capid,Untyped-F1-capid,Typed-F1-reddit,Untyped-F1-reddit
0,v2-bert-capid-det-full-from-ai4-5000-s42,bert,0.5774,0.7044,0.8757,0.8835,0.5523,0.6083
1,v2-distilbert-capid-det-full-from-ai4-15000-s42,distilbert,0.6943,0.7779,0.8728,0.8815,0.4894,0.5477
2,v2-minilm-capid-det-full-from-ai4-5000-s42,minilm,0.6199,0.7266,0.8623,0.8735,0.4574,0.5469


### Which model performs best? - Relevance

#### a) based on capid_validation_f1

In [55]:
metrics = ['accuracy', 'macro_f1']

best_models = comp_plan_rel.sort_values('capid_validation_macro_f1', ascending=False).groupby('model', as_index=False).first()

result = best_models[['run_id', 'model'] + [f'capid_test_{metric}' for metric in metrics] + [f'capid_reddit_{metric}' for metric in metrics]].copy()
result.columns = ['run_id', 'Model', 'accuracy-capid', 'Macro-F1-capid', 'accuracy-reddit', 'Macro-F1-reddit']
result.iloc[:, 2:] = result.iloc[:, 2:].round(4)

display(result)

,run_id,Model,accuracy-capid,Macro-F1-capid,accuracy-reddit,Macro-F1-reddit
0,v2-bert-capid-rel-full-s42,bert,0.8912,0.8745,0.5954,0.5953
1,v2-distilbert-capid-rel-full-s42,distilbert,0.8868,0.8707,0.5449,0.5439
2,v2-minilm-capid-rel-full-s42,minilm,0.8202,0.7921,0.5677,0.5655


#### a) based on capid_test_f1

In [58]:
metrics = ['accuracy', 'macro_f1']

best_models = comp_plan_rel.sort_values('capid_test_macro_f1', ascending=False).groupby('model', as_index=False).first()

result = best_models[['run_id', 'model'] + [f'capid_test_{metric}' for metric in metrics] + [f'capid_reddit_{metric}' for metric in metrics]].copy()
result.columns = ['run_id', 'Model', 'accuracy-capid', 'Macro-F1-capid', 'accuracy-reddit', 'Macro-F1-reddit']
result.iloc[:, 2:] = result.iloc[:, 2:].round(4)

display(result)

,run_id,Model,accuracy-capid,Macro-F1-capid,accuracy-reddit,Macro-F1-reddit
0,v2-bert-capid-rel-906-s42,bert,0.8912,0.8761,0.5726,0.5726
1,v2-distilbert-capid-rel-full-s42,distilbert,0.8868,0.8707,0.5449,0.5439
2,v2-minilm-capid-rel-full-s42,minilm,0.8202,0.7921,0.5677,0.5655


#### b) based on capid_reddit_f1

In [57]:
metrics = ['accuracy', 'macro_f1']

best_models = comp_plan_rel.sort_values('capid_reddit_macro_f1', ascending=False).groupby('model', as_index=False).first()

result = best_models[['run_id', 'model'] + [f'capid_test_{metric}' for metric in metrics] + [f'capid_reddit_{metric}' for metric in metrics]].copy()
result.columns = ['run_id', 'Model', 'accuracy-capid', 'Macro-F1-capid', 'accuracy-reddit', 'Macro-F1-reddit']
result.iloc[:, 2:] = result.iloc[:, 2:].round(4)

display(result)

,run_id,Model,accuracy-capid,Macro-F1-capid,accuracy-reddit,Macro-F1-reddit
0,v2-bert-capid-rel-full-s42,bert,0.8912,0.8745,0.5954,0.5953
1,v2-distilbert-capid-rel-906-s42,distilbert,0.8702,0.8491,0.5644,0.5644
2,v2-minilm-capid-rel-906-s42,minilm,0.7974,0.7647,0.5775,0.5702


### Does more training data help? - Detection

#### a) BERT

In [72]:
metrics = ['span_f1', 'untyped_span_f1']

fix_model = comp_plan_det[(comp_plan_det['model'] == 'bert') & comp_plan_det['source_run_id'].isnull()]
best_models = fix_model.sort_values('ai4privacy_validation_span_f1', ascending=False).groupby('train_size', as_index=False).first()

result = best_models[['run_id', 'model'] + [f'ai4privacy_test_{metric}' for metric in metrics] + [f'capid_test_{metric}' for metric in metrics] + [f'capid_reddit_{metric}' for metric in metrics]].copy()
result.columns = ['run_id', 'Model', 'Typed-F1-ai4', 'Untyped-F1-ai4', 'Typed-F1-capid', 'Untyped-F1-capid', 'Typed-F1-reddit', 'Untyped-F1-reddit']
result.iloc[:, 2:] = result.iloc[:, 2:].round(4)

display(result)

,run_id,Model,Typed-F1-ai4,Untyped-F1-ai4,Typed-F1-capid,Untyped-F1-capid,Typed-F1-reddit,Untyped-F1-reddit
0,v2-bert-capid-det-256-s42,bert,0.2603,0.4229,0.8183,0.8277,0.5128,0.6005
1,v2-bert-capid-det-906-s42,bert,0.2929,0.4550,0.8639,0.8717,0.5435,0.6113
2,v2-bert-capid-det-full-s42,bert,0.3282,0.4773,0.8775,0.8853,0.5445,0.5954
3,v2-bert-ai4-det-5000-s42,bert,0.9515,0.9578,0.2655,0.4087,0.3792,0.4342
4,v2-bert-ai4-det-15000-s42,bert,0.9676,0.9719,0.2485,0.3771,0.2503,0.2877
5,v2-bert-ai4-det-full-s42,bert,0.9759,0.9775,0.2324,0.3709,0.3221,0.3736


#### b) DistilBERT

In [74]:
metrics = ['span_f1', 'untyped_span_f1']

fix_model = comp_plan_det[(comp_plan_det['model'] == 'distilbert') & comp_plan_det['source_run_id'].isnull()]
best_models = fix_model.sort_values('ai4privacy_validation_span_f1', ascending=False).groupby('train_size', as_index=False).first()

result = best_models[['run_id', 'model'] + [f'ai4privacy_test_{metric}' for metric in metrics] + [f'capid_test_{metric}' for metric in metrics] + [f'capid_reddit_{metric}' for metric in metrics]].copy()
result.columns = ['run_id', 'Model', 'Typed-F1-ai4', 'Untyped-F1-ai4', 'Typed-F1-capid', 'Untyped-F1-capid', 'Typed-F1-reddit', 'Untyped-F1-reddit']
result.iloc[:, 2:] = result.iloc[:, 2:].round(4)

display(result)

,run_id,Model,Typed-F1-ai4,Untyped-F1-ai4,Typed-F1-capid,Untyped-F1-capid,Typed-F1-reddit,Untyped-F1-reddit
0,v2-distilbert-capid-det-256-s42,distilbert,0.2235,0.3801,0.7887,0.7988,0.4450,0.5287
1,v2-distilbert-capid-det-906-s42,distilbert,0.2615,0.4125,0.8493,0.8613,0.4751,0.5586
2,v2-distilbert-capid-det-full-s42,distilbert,0.3056,0.4473,0.8867,0.8936,0.4971,0.5674
3,v2-distilbert-ai4-det-5000-s42,distilbert,0.9472,0.9552,0.2702,0.3876,0.3795,0.4246
4,v2-distilbert-ai4-det-15000-s42,distilbert,0.9671,0.9711,0.2440,0.3351,0.3154,0.3535
5,v2-distilbert-ai4-det-full-s42,distilbert,0.9750,0.9776,0.2579,0.3816,0.3705,0.3951


#### c) MiniLM

In [75]:
metrics = ['span_f1', 'untyped_span_f1']

fix_model = comp_plan_det[(comp_plan_det['model'] == 'minilm') & comp_plan_det['source_run_id'].isnull()]
best_models = fix_model.sort_values('ai4privacy_validation_span_f1', ascending=False).groupby('train_size', as_index=False).first()

result = best_models[['run_id', 'model'] + [f'ai4privacy_test_{metric}' for metric in metrics] + [f'capid_test_{metric}' for metric in metrics] + [f'capid_reddit_{metric}' for metric in metrics]].copy()
result.columns = ['run_id', 'Model', 'Typed-F1-ai4', 'Untyped-F1-ai4', 'Typed-F1-capid', 'Untyped-F1-capid', 'Typed-F1-reddit', 'Untyped-F1-reddit']
result.iloc[:, 2:] = result.iloc[:, 2:].round(4)

display(result)

,run_id,Model,Typed-F1-ai4,Untyped-F1-ai4,Typed-F1-capid,Untyped-F1-capid,Typed-F1-reddit,Untyped-F1-reddit
0,v2-minilm-capid-det-256-s42,minilm,0.1241,0.2763,0.6718,0.7592,0.3828,0.5118
1,v2-minilm-capid-det-906-s42,minilm,0.2821,0.4438,0.8443,0.8590,0.4503,0.5430
2,v2-minilm-capid-det-full-s42,minilm,0.2907,0.4287,0.8612,0.8699,0.4203,0.4979
3,v2-minilm-ai4-det-5000-s42,minilm,0.9396,0.9487,0.2387,0.3982,0.3473,0.4092
4,v2-minilm-ai4-det-15000-s42,minilm,0.9630,0.9686,0.2459,0.3838,0.3448,0.4054
5,v2-minilm-ai4-det-full-s42,minilm,0.9733,0.9754,0.2413,0.3719,0.3286,0.3787
